<a href="https://colab.research.google.com/github/shukhratbekalijonov5-oss/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu Detector")

Menu Detector


In [ ]:
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2
import torchvision.transforms as transforms

# PIL = python image library
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Dataset main folder path in Google Drive
# This folder contains our food image folders:
# cheesecake, chocolate_cake, hot_dog, kebab, pilaf, pizza

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("\nDataset path:", DATASET_PATH)


# Label grouping / class consolidation
# This dictionary maps original folder names to final class names.
#
# Format:
# "original_folder_name": "final_class_name"
#
# Here, cheesecake and chocolate_cake will be grouped as one class: dessert

CUSTOM_CLASS_MAPPING = {
    'cheesecake': 'dessert',
    'chocolate_cake': 'dessert',
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'kebab': 'kebab',
    'pilaf': 'pilaf',
    'pizza': 'pizza'
}


# Final class names that our model will learn
# cheesecake and chocolate_cake are not separate classes now.
# They are combined into one class: dessert

CLASSES = [
    'dessert',
    'hamburger',
    'hot_dog',
    'kebab',
    'pilaf',
    'pizza'
]
print("\nClasses:", CLASSES)


# Count how many final classes we have
NUM_CLASSES = len(CLASSES)
print("\nClass length:", NUM_CLASSES)


# Convert class names into numbers
# AI models work with numbers, not text labels.
#
# Result will be:
# {
#   'dessert': 0,
#   'hamburger': 1,
#   'hot_dog': 2,
#   'kebab': 3,
#   'pilaf': 4,
#   'pizza': 5
# }

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
print("\nCLASS INDEX:", CLASS_TO_IDX)

# ------------
# Create image transformation pipeline
# This pipeline prepares each image before giving it to the model

transform = transforms.Compose([

    # Resize every image to 224x224 pixels
    # Models need images with the same size
    transforms.Resize((224, 224)),

    # Convert image to PyTorch tensor
    # Also changes pixel values from 0-255 to 0.0-1.0
    transforms.ToTensor(),

    # Normalize image pixel values for each RGB channel
    # Formula: new_value = (old_value - mean) / std
    #
    # mean and std values are commonly used for pretrained ImageNet models
    #
    # Red channel:   (value - 0.485) / 0.229
    # Green channel: (value - 0.456) / 0.224
    # Blue channel:  (value - 0.406) / 0.225
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Dataset path: /content/drive/MyDrive/food101_dataset

Classes: ['dessert', 'hamburger', 'hot_dog', 'kebab', 'pilaf', 'pizza']

Class length: 6

CLASS INDEX: {'dessert': 0, 'hamburger': 1, 'hot_dog': 2, 'kebab': 3, 'pilaf': 4, 'pizza': 5}


CUSTOM_CLASS_MAPPING -> rename/group folder labels

CLASSES -> final categories for the model

CLASS_TO_IDX -> convert class names to numbers for model training

In [ ]:
#------------------------------
# Custom Dataset Class
#------------------------------

# FoodDataset class inherits from PyTorch Dataset
# This class helps us load images and labels one by one for model training

class FoodDataset(Dataset):

    # __init__ runs when we create dataset object
    # images -> list of image file paths
    # labels -> list of image labels
    # transform -> image preprocessing pipeline, for example Resize, ToTensor, Normalize
    def __init__(self, images, labels, transform=None):
        self.images = images          # Store image paths
        self.labels = labels          # Store labels
        self.transform = transform    # Store transform pipeline
        # Store will not write index: "0": "image_path_1" | it writes only "image_path_1"
        # Python list automatically has index.
        # Store shape:
        # images = [image_path_1, image_path_2, image_path_3]
        # labels = [label_1, label_2, label_3]


    # __len__ returns total number of images in dataset
    # PyTorch uses this to know dataset size
    def __len__(self):
        print('images_length:', len(self.images))
        return len(self.images)


    # __getitem__ gets one image and its label by index
    # PyTorch DataLoader calls this method automatically
    def __getitem__(self, idx):

        # Get image path by index
        img_path = self.images[idx]
        print("image_path:", img_path)


        # Get label by same index
        # Example:
        # image path -> images[0]
        # label      -> labels[0]
        label = self.labels[idx]
        print('label:', label)


        # Try to open image file
        # convert('RGB') makes sure image has 3 color channels: Red, Green, Blue
        try:
            image = Image.open(img_path).convert('RGB')


        # If image is broken or cannot be opened, skip it
        # Then load next image instead
        except (UnidentifiedImageError, OSError):
            print(f'Skipping broken image: {img_path}')
            return self.__getitem__((idx + 1) % len(self.images))


        # If transform exists, apply it to the image
        # Example:
        # Resize -> ToTensor -> Normalize
        if self.transform:
            image = self.transform(image)


        # Return final image and label
        # This is what model receives during training
        return image, label

In [ ]:
# ============================
# Gather and Split Data
# ============================

# This list will store all image paths with their labels
# Each item will look like this:
# ("image_path", label_number)
all_images = []


# Loop through custom class mapping
# original_class -> folder name in Google Drive         | cheesecake
# mapped_class   -> final class name used by the model. | dessert
#   cheesecake      dessert        {'cheesecake': 'dessert'.....}
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():

    # Create full path for each class folder
    # Example:
    # /content/drive/MyDrive/food101_dataset/cheesecake
    class_path = os.path.join(DATASET_PATH, original_class)
    print("class_path:", class_path)

    # Check if this class folder exists
    # If folder does not exist, skip it
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue


    # Loop through all image files inside the class folder
    for img in os.listdir(class_path):

        # Only use image files with these extensions
        if img.endswith(('.jpg', '.jpeg', '.png')):

            # Create full image path
            # Example:
            # /content/drive/MyDrive/food101_dataset/pizza/image1.jpg
            full_path = os.path.join(class_path, img)

            # Get label number from mapped class name
            # Example:
            # mapped_class = "pizza"
            # CLASS_TO_IDX["pizza"]  | label = 4
            label = CLASS_TO_IDX[mapped_class]

            # Add image path and label together
            # This keeps image and label matched by same index
            all_images.append((full_path, label))


# Shuffle all images randomly
# This helps mix classes before splitting train/validation data
np.random.shuffle(all_images)

# Split 80% for training and 20% for validation
split = int(0.8 * len(all_images))

# First 80% data for training
train_data = all_images[:split]

# Last 20% data for validation
val_data = all_images[split:]


# Separate image paths and labels for training data
# train_images -> image paths
# train_labels -> label numbers
train_images, train_labels = zip(*train_data)

# Separate image paths and labels for validation data
# val_images -> image paths
# val_labels -> label numbers
val_images, val_labels = zip(*val_data)


# Print all image-label pairs to check
print("first 3 images:", all_images[:3])


# Create dataset object with training (image paths |and| labels)
# FoodDataset stores:
# self.images = train_images
# self.labels = train_labels
dataset = FoodDataset(train_images, train_labels)

# Print total number of training images
# This calls __len__()
print(len(dataset))

# Get one image-label pair by index
# 291 is item index, not label number
# This calls __getitem__(291)
#
# Example:
# dataset[291] -> self.images[291] + self.labels[291]
img, lbl = dataset[291]

class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
class_path: /content/drive/MyDrive/food101_dataset/pizza
first 3 images: [('/content/drive/MyDrive/food101_dataset/cheesecake/1853531.jpg', 0), ('/content/drive/MyDrive/food101_dataset/hot_dog/111028.jpg', 2), ('/content/drive/MyDrive/food101_dataset/cheesecake/1626857.jpg', 0)]
images_length: 4160
4160
image_path: /content/drive/MyDrive/food101_dataset/hot_dog/2598934.jpg
label: 2


In [ ]:
# Create training and validation Dataset objects
# At this point, they only store image paths, labels, and transform.
# They do NOT open/read images yet.
#
# Images are loaded later when we call:
# train_dataset[index] or val_dataset[index]

train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

# train_dataset contains:
# image paths + labels + transform rule

In [ ]:
# Create DataLoader for training dataset
# DataLoader gets samples from train_dataset and groups them into batches
# DataLoader = Take data from train_dataset -> but give it to me 32 images at a time.
#
# batch_size=32 -> each batch contains 32 images and 32 labels
# shuffle=True  -> mix training data every epoch, good for training
# num_workers=2 -> use 2 background workers/threads to load data faster

# train_dataset -> stores data and loads one item | train_dataset[0] -> image_0 + label_0
# train_loader  -> loads many items as batches for training | loop train_loader -> 32 images + 32 labels

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

images_length: 4160
images_length: 4160


In [ ]:
# Load pretrained MobileNetV2 model
# pretrained means the model was already trained on ImageNet dataset
# So we do not build and train a CNN from zero
#
# mobilenet_v2 -> ready CNN model for image classification
# weights="IMAGENET1K_V1" means:
# use pretrained weights from ImageNet dataset
model = mobilenet_v2(weights="IMAGENET1K_V1")


# MobileNetV2 classifier has 2 parts:
# model.classifier[0] -> Dropout layer | Dropout helps reduce overfitting during training.
# model.classifier[1] -> Linear layer  | Linear layer is the final prediction layer.
#

# MobileNetV2 already learned useful image features
# Example features:
# edges, colors, textures, shapes, object parts
#
# But original MobileNetV2 was trained for 1000 ImageNet classes.
# Our project has only NUM_CLASSES classes.
#
# model.classifier[1] is the final prediction layer.
# model.classifier[1].in_features is the number of features coming into this layer.
# For MobileNetV2, it is usually 1280.
#
# We replace the old final layer:
# 1280 features -> 1000 ImageNet classes
#
# With our new final layer:
# 1280 features -> NUM_CLASSES food classes

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,  # input features from MobileNetV2
    NUM_CLASSES                       # output classes for our dataset
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 103MB/s] 


In [ ]:
# Select device for model training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to selected device
# Model and data must be on the same device during training
# Example:
# model on GPU + images on GPU -> correct
# model on CPU + images on GPU -> error
model = model.to(device)

Using device: cuda


In [ ]:
# Loss function
# CrossEntropyLoss is commonly used for multi-class classification.
#
# It compares:
# model prediction scores vs correct labels
#
# Example:
# prediction -> [score_for_dessert, score_for_hot_dog, score_for_kebab, ...]
# label      -> correct class number
#
# Loss tells us how wrong the model prediction is.
criterion = nn.CrossEntropyLoss()


# Optimizer
# Adam optimizer updates model parameters during training.
#
# model.parameters() -> all trainable weights/biases inside the model
# lr=0.001 -> learning rate, controls how big each update step is
#
# Simple meaning:
# optimizer -> changes model weights to reduce loss
optimizer = optim.Adam(model.parameters(), lr=0.001)


# cuDNN benchmark setting
# This can make training faster on GPU.
#
# When input image size is fixed, for example 224x224,
# PyTorch can search and use the fastest GPU algorithm.
#
# Good for fixed image sizes.
# May not help if image sizes keep changing.
torch.backends.cudnn.benchmark = True